# 03 — Grover's Search Algorithm

**Quantum Information Processing** | p-engel & collaborator

---

Grover's algorithm finds a marked item in an **unstructured database of N items** in O(√N) queries — a quadratic speedup over any classical algorithm.

### Core idea
1. Start in equal superposition of all states
2. **Oracle** — flips the phase of the target state(s): |x⟩ → -|x⟩ if x is the answer
3. **Diffuser** (inversion about the mean) — amplifies the target amplitude
4. Repeat ~π/4 · √N times, then measure

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram
from qiskit.quantum_info import Statevector

## 1. Build a 2-qubit Grover circuit (4 items, 1 marked)

Search space: {|00⟩, |01⟩, |10⟩, |11⟩} 
Target: |11⟩ (we'll tell the oracle to mark this)

In [ ]:
def grover_oracle_11(n_qubits=2):
    """Oracle that marks the state |11⟩ by flipping its phase."""
    oracle = QuantumCircuit(n_qubits)
    # CZ gate flips phase of |11⟩
    oracle.cz(0, 1)
    oracle.name = "Oracle |11⟩"
    return oracle


def grover_diffuser(n_qubits):
    """Inversion about the mean (Grover diffuser)."""
    diffuser = QuantumCircuit(n_qubits)
    diffuser.h(range(n_qubits))
    diffuser.x(range(n_qubits))
    diffuser.h(n_qubits - 1)
    diffuser.mcx(list(range(n_qubits - 1)), n_qubits - 1)  # multi-controlled X
    diffuser.h(n_qubits - 1)
    diffuser.x(range(n_qubits))
    diffuser.h(range(n_qubits))
    diffuser.name = "Diffuser"
    return diffuser


def build_grover(n_qubits, n_iterations, oracle):
    """Full Grover circuit: init → (oracle + diffuser) × n_iterations → measure."""
    qc = QuantumCircuit(n_qubits, n_qubits)
    
    # Initialize: equal superposition
    qc.h(range(n_qubits))
    qc.barrier()
    
    # Grover iterations
    for _ in range(n_iterations):
        qc.append(oracle.to_gate(), range(n_qubits))
        qc.append(grover_diffuser(n_qubits).to_gate(), range(n_qubits))
        qc.barrier()
    
    qc.measure(range(n_qubits), range(n_qubits))
    return qc


n = 2
N = 2**n  # 4 items
optimal_iters = int(np.round(np.pi / 4 * np.sqrt(N)))  # ≈ 1 for N=4
print(f"N = {N} items, optimal iterations = {optimal_iters}")

oracle = grover_oracle_11(n)
grover_qc = build_grover(n, optimal_iters, oracle)
grover_qc.draw('mpl')

## 2. Simulate and visualize

In [ ]:
simulator = AerSimulator()
compiled = transpile(grover_qc, simulator)
counts = simulator.run(compiled, shots=2048).result().get_counts()

print("Counts:", counts)
plot_histogram(counts, title="Grover's Algorithm: target = |11⟩")

**Expected:** |11⟩ dominates with very high probability (~100% for N=4, 1 iteration).

## 3. Amplitude evolution — watch the probability grow

Here we track how the target state's amplitude grows with each Grover iteration.

In [ ]:
max_iters = 6
target_probs = []

for k in range(1, max_iters + 1):
    qc_k = build_grover(n, k, grover_oracle_11(n))
    # Use statevector (no measurement) to get exact probabilities
    qc_sv = qc_k.remove_final_measurements(inplace=False)
    sv = Statevector(qc_sv)
    probs = sv.probabilities_dict()
    target_probs.append(probs.get('11', 0))

iters = np.arange(1, max_iters + 1)
theory = np.sin((2 * iters + 1) * np.arcsin(1 / np.sqrt(N)))**2

plt.figure(figsize=(7, 4))
plt.plot(iters, target_probs, 'o-', label='Simulated P(|11⟩)', lw=2)
plt.plot(iters, theory, '--', label='Theory: sin²((2k+1)·arcsin(1/√N))', alpha=0.7)
plt.xlabel('Number of Grover iterations')
plt.ylabel('P(target state)')
plt.title("Amplitude Amplification: N=4, target=|11⟩")
plt.ylim(0, 1.05)
plt.xticks(iters)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Scale up — 3 qubits (N=8)

Optimal iterations: ~2 for N=8.

In [ ]:
def grover_oracle_111(n_qubits=3):
    """Marks state |111⟩."""
    oracle = QuantumCircuit(n_qubits)
    oracle.h(n_qubits - 1)
    oracle.mcx(list(range(n_qubits - 1)), n_qubits - 1)
    oracle.h(n_qubits - 1)
    oracle.name = "Oracle |111⟩"
    return oracle

n3 = 3
N3 = 2**n3
k3 = int(np.round(np.pi / 4 * np.sqrt(N3)))
print(f"N={N3}, optimal iters={k3}")

grover3 = build_grover(n3, k3, grover_oracle_111(n3))
compiled3 = transpile(grover3, simulator)
counts3 = simulator.run(compiled3, shots=4096).result().get_counts()

plot_histogram(counts3, title="Grover's Algorithm: 3 qubits, target=|111⟩")

## 5. Connection to our research

Grover-style amplitude amplification is a subroutine in many quantum algorithms. In the context of quantum information processing with photonic systems:

- **Oracle** → encodes a problem constraint (e.g., selecting a resonance condition)
- **Amplitude amplification** → underpins quantum speedup in combinatorial search
- Related to **boson sampling** and linear optical quantum computing (LOQC), where photons serve as natural qubits

### Next: Quantum Fourier Transform (QFT)
The QFT is the quantum analog of the DFT and appears in Shor's algorithm, phase estimation, and quantum signal processing — all relevant to our work on spectral and photonic properties.